# 02 — EDA: EU Storage Levels
Exploratory analysis of EU aggregate gas storage.

In [ ]:
import sys, os, warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
warnings.filterwarnings('ignore')

for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / 'src' / 'agsi_client').exists():
        sys.path.insert(0, str(_c))
        os.chdir(_c)
        print(f"Project root: {_c}")
        break

DATA = Path("data/processed")
df = pd.read_parquet(DATA / "eu_aggregate_full.parquet")
df.index = pd.to_datetime(df.index).tz_localize(None)
df = df.sort_index()
df["year"] = df.index.year
df["month"] = df.index.month
df["doy"] = df.index.day_of_year
print(f"eu_aggregate_full.parquet: {df.shape}")
print(f"  {df.index[0].date()} -> {df.index[-1].date()}")
print(f"  Columns: {list(df.columns)}")


## 1. Full History of EU Storage Fill %

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df.index, y=df["full"],
    name="EU Fill %", line=dict(color="#2E75B6", width=1.5)
))
fig.add_hline(y=90, line_dash="dot", line_color="purple",
              annotation_text="90% EU Target", annotation_position="top right")
fig.add_hline(y=20, line_dash="dot", line_color="red",
              annotation_text="20% Critical", annotation_position="bottom right")
fig.update_layout(
    title="EU Gas Storage Fill % — Full History",
    xaxis_title="Date", yaxis_title="Fill (%)",
    yaxis=dict(range=[0, 110]),
    height=450, template="plotly_white"
)
fig.show()


## 2. Distribution of EU Daily Fill %

In [ ]:
fig = px.histogram(
    x=df["full"].dropna(),
    nbins=50,
    title="Distribution of EU Daily Storage Fill %",
    labels={"x": "Fill (%)", "count": "Days"},
    color_discrete_sequence=["#2E75B6"]
)
med = df["full"].median()
mn  = df["full"].mean()
fig.add_vline(x=med, line_dash="dash", line_color="orange",
              annotation_text=f"Median: {med:.1f}%", annotation_position="top right")
fig.add_vline(x=mn,  line_dash="dash", line_color="green",
              annotation_text=f"Mean: {mn:.1f}%", annotation_position="top left")
fig.update_layout(height=400, template="plotly_white", showlegend=False)
fig.show()
print(f"Mean:   {mn:.1f}%  |  Median: {med:.1f}%  |  Std: {df['full'].std():.1f}%")
print(f"Min:    {df['full'].min():.1f}%  |  Max:    {df['full'].max():.1f}%")


## 3. Year-over-Year Overlay
Each year's fill curve plotted on the same day-of-year axis.

In [ ]:
import plotly.colors as pc
colors = pc.qualitative.Set1 + pc.qualitative.Set2
fig = go.Figure()
years = sorted(df["year"].unique())
for i, yr in enumerate(years):
    yr_data = df[df["year"] == yr].copy()
    lw      = 2.5 if yr == years[-1] else 1.0
    opacity = 1.0 if yr >= years[-1] - 3 else 0.35
    fig.add_trace(go.Scatter(
        x=yr_data["doy"], y=yr_data["full"],
        name=str(yr), mode="lines",
        line=dict(color=colors[i % len(colors)], width=lw),
        opacity=opacity
    ))
fig.add_hline(y=90, line_dash="dot", line_color="black", line_width=1,
              annotation_text="90% Target")
fig.update_layout(
    title="EU Gas Storage — Year-over-Year Overlay (same day-of-year axis)",
    xaxis=dict(
        title="Day of Year",
        tickvals=[1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335],
        ticktext=["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    ),
    yaxis=dict(title="Fill (%)", range=[0, 110]),
    height=500, template="plotly_white"
)
fig.show()


## 4. Summary Statistics by Year

In [ ]:
stats = (
    df.groupby("year")["full"]
    .agg(Min="min", Max="max", Mean="mean", Std="std")
    .round(1)
)
winter_low = df[df["month"].isin([1,2,3])].groupby("year")["full"].min().round(1)
summer_peak = df[df["month"].isin([8,9,10])].groupby("year")["full"].max().round(1)
stats = stats.assign(**{"Winter Low (Jan-Mar)": winter_low, "Summer Peak (Aug-Oct)": summer_peak})
print("EU Gas Storage Fill % — Annual Summary")
print("=" * 65)
print(stats.to_string())
